In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp

from estimator import MyEstimator

from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from itertools import product

import hamiltonian_generator, evolution_circuits

import warnings
warnings.filterwarnings("ignore")

In [2]:
service = QiskitRuntimeService()
backend_name = 'ibm_kingston'
backend = service.backend(backend_name)

qiskit_runtime_service.__init__:WARNING:2026-04-28 08:01:28,678: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: m5000-eu, m5000-us. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-28 08:01:28,679: Using instance: m5000-us, plan: premium


In [3]:
sampler = Sampler(backend)

In [4]:
num_sites = [5, 12]
couplings = [0]
dd_enabled = [False, True]
folds = [1, 3, 5]

final_time = 0.25
num_timesteps = 5

In [5]:
for (n, h, dd, f) in product(num_sites, couplings, dd_enabled, folds) :
    dd_str = ('dd' if dd else 'std')
    job_name = f'evo_job_{n}_{h}_{dd_str}_{f}'

    circuit = evolution_circuits.dynamic_evolve(
        n, h, final_time, num_timesteps, mcm=True, stretch_dd=dd, folds=f)

    dynamic_hamiltonian = hamiltonian_generator.get_dynamic_ising_hamiltonian(n, h)

    estimator = MyEstimator(
        job_name, circuit, None, dynamic_hamiltonian, sampler, service, backend)

    print(f'Submitting {job_name}')
    estimator.submit_sampler_jobs()

Submitting evo_job_5_0_std_1
Submitting evo_job_5_0_std_3
Submitting evo_job_5_0_std_5
Submitting evo_job_5_0_dd_1
Submitting evo_job_5_0_dd_3
Submitting evo_job_5_0_dd_5
Submitting evo_job_12_0_std_1
Submitting evo_job_12_0_std_3
Submitting evo_job_12_0_std_5
Submitting evo_job_12_0_dd_1
Submitting evo_job_12_0_dd_3
Submitting evo_job_12_0_dd_5


In [6]:
for (n, h, dd, f) in product(num_sites, couplings, dd_enabled, folds) :
    dd_str = ('dd' if dd else 'std')
    job_name = f'evo_job_{n}_{h}_{dd_str}_{f}'

    circuit = evolution_circuits.dynamic_evolve(
        n, h, final_time, num_timesteps, mcm=True, stretch_dd=dd, folds=f)

    dynamic_hamiltonian = hamiltonian_generator.get_dynamic_ising_hamiltonian(n, h)

    estimator = MyEstimator(
        job_name, circuit, None, dynamic_hamiltonian, sampler, service, backend)

    print(f'Evaluating {job_name}')
    energy = estimator.evaluate_expectation_value()
    print(f"{job_name}: \t\t {energy}")
    np.save(f'data/evo_energy_{n}_{h}_{dd_str}_{f}', energy)

Evaluating evo_job_5_0_std_1
evo_job_5_0_std_1: 		 3.92431640625
Evaluating evo_job_5_0_std_3
evo_job_5_0_std_3: 		 3.88916015625
Evaluating evo_job_5_0_std_5
evo_job_5_0_std_5: 		 3.884765625
Evaluating evo_job_5_0_dd_1
evo_job_5_0_dd_1: 		 3.3076171875
Evaluating evo_job_5_0_dd_3
evo_job_5_0_dd_3: 		 2.5146484375
Evaluating evo_job_5_0_dd_5
evo_job_5_0_dd_5: 		 2.04052734375
Evaluating evo_job_12_0_std_1
evo_job_12_0_std_1: 		 10.73291015625
Evaluating evo_job_12_0_std_3
evo_job_12_0_std_3: 		 9.6552734375
Evaluating evo_job_12_0_std_5
evo_job_12_0_std_5: 		 9.677734375
Evaluating evo_job_12_0_dd_1
evo_job_12_0_dd_1: 		 9.07666015625
Evaluating evo_job_12_0_dd_3
evo_job_12_0_dd_3: 		 6.330078125
Evaluating evo_job_12_0_dd_5
evo_job_12_0_dd_5: 		 5.25
